# What \"Generative\" Means: Autoregression, the Chain Rule, and Perplexity

The runnable code of [What \"Generative\" Means: Autoregression, the Chain Rule, and Perplexity](https://residual-stream.lvtan71.workers.dev/posts/autoregression-and-perplexity/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/autoregression-and-perplexity.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 22 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "autoregression-and-perplexity"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/autoregression-and-perplexity/lm.py', 'data/blog_train.txt', 'data/blog_val.txt']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## One line of probability

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/autoregression-and-perplexity/#one-line-of-probability)

In [ ]:
import math
from collections import Counter

# The chain rule, with the simplest model that obeys it: each character's
# probability given only the one before it (a bigram model), counted on the
# blog's training text with add-one smoothing.
train, val = open("blog_train.txt").read(), open("blog_val.txt").read()
V = len(set(train)) + 1
first, pair = Counter(train), Counter(train[i:i + 2] for i in range(len(train) - 1))
p = lambda c, prev: (pair[prev + c] + 1) / (first[prev] + V)

text = "the cache"
logp = math.log(first[text[0]] / len(train))
print(f"p('{text[0]}') = {math.exp(logp):.4f}")
for prev, c in zip(text, text[1:]):
    logp += math.log(p(c, prev))
    print(f"p('{c}' | '{prev}') = {p(c, prev):.4f}")
print(f"p('{text}') = product = {math.exp(logp):.3e};  -log2 p = {-logp / math.log(2):.1f} bits "
      f"for {len(text)} characters")

# Perplexity of the bigram model on the held-out posts.
nll = -sum(math.log(p(c, prev)) for prev, c in zip(val, val[1:])) / (len(val) - 1)
print(f"bigram model on held-out posts: {nll:.3f} nats per character, perplexity {math.exp(nll):.1f}, "
      f"{nll / math.log(2):.3f} bits per character")

The post printed:

```text
p('t') = 0.0635
p('h' | 't') = 0.2066
p('e' | 'h') = 0.4248
p(' ' | 'e') = 0.2166
p('c' | ' ') = 0.0259
p('a' | 'c') = 0.1363
p('c' | 'a') = 0.0674
p('h' | 'c') = 0.1844
p('e' | 'h') = 0.4248
p('the cache') = product = 2.250e-08;  -log2 p = 25.4 bits for 9 characters
bigram model on held-out posts: 2.849 nats per character, perplexity 17.3, 4.110 bits per character
```

## Training: teacher forcing

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/autoregression-and-perplexity/#training-teacher-forcing)

In [ ]:
%%writefile lm.py
import math
import sys
import torch
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

# An LSTM language model of the blog, over characters or over byte-level BPE
# tokens, trained and scored the same way: predict each next unit from the
# ones before it. The BPE tokenizer is trained on the training text alone
# (shown in "Byte-Pair Encoding"), with every one of the 256 bytes in its vocabulary, so
# any held-out text can be written in its tokens.
torch.set_num_threads(4)
train_text, val_text = open("blog_train.txt").read(), open("blog_val.txt").read()
bpe = Tokenizer(models.BPE())
bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe.train_from_iterator([train_text], trainers.BpeTrainer(
    vocab_size=2048, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))

def units(text, kind):
    return list(text) if kind == "char" else bpe.encode(text).ids

class LSTMLM(torch.nn.Module):
    def __init__(self, V, hidden=256):
        super().__init__()
        self.embed = torch.nn.Embedding(V, 64)
        self.rnn = torch.nn.LSTM(64, hidden, batch_first=True)
        self.out = torch.nn.Linear(hidden, V)
    def forward(self, x, state=None):
        h, state = self.rnn(self.embed(x), state)
        return self.out(h), state

def build(kind):
    """Units as tensors. Characters never seen in training share id 0; BPE
    tokens use the tokenizer's own ids, all of which exist from the start."""
    tr, va = units(train_text, kind), units(val_text, kind)
    if kind == "char":
        vocab = sorted(set(tr))
        index = {u: i + 1 for i, u in enumerate(vocab)}
        to_t = lambda us: torch.tensor([index.get(u, 0) for u in us])
        return vocab, to_t(tr), to_t(va)
    return list(range(bpe.get_vocab_size() - 1)), torch.tensor(tr), torch.tensor(va)

@torch.no_grad()
def total_nats(model, data, T=128):
    """Sum of -log p over every unit of `data`, carrying the state along."""
    total, state = 0.0, None
    for s in range(0, len(data) - 1, T):
        x, y = data[s:s + T], data[s + 1:s + T + 1]
        logits, state = model(x[None][:, :len(y)], state)
        total += torch.nn.functional.cross_entropy(logits[0], y, reduction="sum").item()
    return total

def train(kind, seed, steps=3000, B=64, T=128, checkpoints=()):
    """Returns the model and, at each checkpoint, validation bits per byte."""
    vocab, tr, va = build(kind)
    n_bytes, seen = len(val_text.encode("utf-8")), {}
    torch.manual_seed(seed)
    model, g = LSTMLM(len(vocab) + 1), torch.Generator().manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.0)
    for step in range(1, steps + 1):
        i = torch.randint(0, len(tr) - T - 1, (B,), generator=g)
        x = torch.stack([tr[j:j + T] for j in i]); y = torch.stack([tr[j + 1:j + T + 1] for j in i])
        logits, _ = model(x)                                  # teacher forcing: every position at once
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step in checkpoints:
            seen[step] = total_nats(model, va) / math.log(2) / n_bytes
    return model, vocab, va, seen

if __name__ == "__main__":
    kind, seed = sys.argv[1], int(sys.argv[2])
    model, vocab, va, seen = train(kind, seed, checkpoints=(250, 500, 1000, 2000, 3000))
    tr = build(kind)[1]
    unseen = (~torch.isin(va, tr)).float().mean().item()      # held-out units absent from training
    nats = total_nats(model, va)
    per_unit = nats / (len(va) - 1)
    passes = 3000 * 64 * 128 / len(tr)
    params = sum(p.numel() for p in model.parameters())
    print(f"{kind:4s} seed {seed}: {len(vocab) + 1:,} units, {params:,} parameters, "
          f"{len(val_text.encode()) / len(va):.2f} bytes per unit, {unseen:.1%} of held-out units unseen "
          f"in training; at 3,000 steps ({passes:.0f} passes) perplexity {math.exp(per_unit):.1f} per unit")
    print("      validation bits per byte at 250 / 500 / 1,000 / 2,000 / 3,000 steps: "
          + " / ".join(f"{v:.3f}" for v in seen.values()))
    if kind == "char" and seed == 0:
        torch.save({"model": model.state_dict(), "vocab": vocab}, "char_lstm.pt")

In [ ]:
# The post trained each unit (characters, BPE tokens) with two seeds:
for kind in ("char", "bpe"):
    for seed in (0, 1):
        %run lm.py {kind} {seed}

The post printed:

```text
char seed 0: 148 units, 377,236 parameters, 1.01 bytes per unit, 0.2% of held-out units unseen in training; at 3,000 steps (41 passes) perplexity 6.4 per unit
      validation bits per byte at 250 / 500 / 1,000 / 2,000 / 3,000 steps: 3.311 / 2.973 / 2.737 / 2.621 / 2.657
char seed 1: 148 units, 377,236 parameters, 1.01 bytes per unit, 0.2% of held-out units unseen in training; at 3,000 steps (41 passes) perplexity 6.4 per unit
      validation bits per byte at 250 / 500 / 1,000 / 2,000 / 3,000 steps: 3.297 / 2.967 / 2.741 / 2.647 / 2.659
bpe  seed 0: 2,048 units, 987,136 parameters, 2.56 bytes per unit, 0.6% of held-out units unseen in training; at 3,000 steps (113 passes) perplexity 261.5 per unit
      validation bits per byte at 250 / 500 / 1,000 / 2,000 / 3,000 steps: 2.912 / 2.730 / 2.692 / 2.853 / 3.133
bpe  seed 1: 2,048 units, 987,136 parameters, 2.56 bytes per unit, 0.6% of held-out units unseen in training; at 3,000 steps (113 passes) perplexity 268.9 per unit
      validation bits per byte at 250 / 500 / 1,000 / 2,000 / 3,000 steps: 2.903 / 2.728 / 2.690 / 2.865 / 3.148
```

## Generation: the factorization run forward

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/autoregression-and-perplexity/#generation-the-factorization-run-forward)

In [ ]:
import time
import torch
from lm import LSTMLM

# Generation is the factorization run forward: predict a distribution for the
# next character, pick one, append it, repeat.
torch.set_num_threads(4)
saved = torch.load("char_lstm.pt")
vocab = saved["vocab"]
index = {c: i + 1 for i, c in enumerate(vocab)}
model = LSTMLM(len(vocab) + 1)
model.load_state_dict(saved["model"]); model.eval()

@torch.no_grad()
def generate(prompt, n, greedy, seed=0):
    g = torch.Generator().manual_seed(seed)
    logits, state = model(torch.tensor([[index[c] for c in prompt]]))
    out = []
    for _ in range(n):
        p = torch.softmax(logits[0, -1], -1)
        nxt = p.argmax() if greedy else torch.multinomial(p, 1, generator=g)[0]
        out.append(vocab[nxt - 1] if nxt > 0 else "?")
        logits, state = model(nxt.view(1, 1), state)       # feed its own output back in
    return "".join(out)

prompt = "The KV cache "
print(f"greedy:  {prompt}{generate(prompt, 200, greedy=True)!r}")
print(f"sampled: {prompt}{generate(prompt, 200, greedy=False)!r}")

# Three ways through 512 characters, median of 7 runs each: scoring known
# text in one call (teacher forcing), feeding the same known text one call
# per character, and generating (each step waits for the character just chosen).
text = open("blog_val.txt").read()[:513]
x = torch.tensor([[index.get(c, 0) for c in text[:-1]]])

def median_ms(fn, runs=7):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return 1e3 * sorted(times)[runs // 2]

@torch.no_grad()
def one_call_per_char():
    state = None
    for t in range(x.shape[1]):
        _, state = model(x[:, t:t + 1], state)

with torch.no_grad():
    forced = median_ms(lambda: model(x))
stepwise, generated = median_ms(one_call_per_char), median_ms(lambda: generate("T", 512, greedy=True))
print(f"512 characters: teacher forcing {forced:.1f} ms; the same known characters one call at a time "
      f"{stepwise:.1f} ms; generating {generated:.1f} ms")

The post printed:

```text
greedy:  The KV cache 'post costs a settings of the same post is the same post is the same post is the same post is the same post is the same post is the same post 2.1 on the same post 2.1 found the same post 2.1 on the sam'
sampled: The KV cache 'han the routure model. Write at it\n*sort** (Zheng](hts forkens only a\nmeany sublay.\n- Imm and generated units including token\n  to best process to the graming story, and every step. The loss over\n  us'
512 characters: teacher forcing 5.9 ms; the same known characters one call at a time 16.4 ms; generating 19.2 ms
```

## Perplexity, and why it depends on the tokenizer

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/autoregression-and-perplexity/#perplexity-and-why-it-depends-on-the-tokenizer)

In [ ]:
import bz2
import lzma
import math
import zlib
from collections import Counter

# Baselines for "bits per byte" on the held-out posts, from knowing nothing
# to general-purpose compressors that are allowed to read the training posts.
train, val = open("blog_train.txt").read(), open("blog_val.txt").read()
tb, vb = train.encode(), val.encode()
n = len(vb)
print(f"raw UTF-8 bytes:                         {8.0:.2f} bits per byte")
V = len(set(train)) + 1
print(f"uniform guess over {V} characters:        {math.log2(V) * len(val) / n:.2f}")
freq = Counter(train)
uni = -sum(math.log2((freq[c] + 1) / (len(train) + V)) for c in val) / n
print(f"unigram character counts:                {uni:.2f}")
for name, f in [("zlib -9", lambda b: zlib.compress(b, 9)), ("bz2 -9", lambda b: bz2.compress(b, 9)),
                ("xz -9e", lambda b: lzma.compress(b, preset=9 | lzma.PRESET_EXTREME))]:
    alone = 8 * len(f(vb)) / n
    given = 8 * (len(f(tb + vb)) - len(f(tb))) / n       # extra bytes to add the held-out posts
    print(f"{name:8s} on the held-out posts alone {alone:.2f}; given the training posts {given:.2f}")

The post printed:

```text
raw UTF-8 bytes:                         8.00 bits per byte
uniform guess over 148 characters:        7.13
unigram character counts:                4.95
zlib -9  on the held-out posts alone 3.15; given the training posts 3.03
bz2 -9   on the held-out posts alone 2.87; given the training posts 2.34
xz -9e   on the held-out posts alone 2.95; given the training posts 2.44
```